In [1]:
!pip install pyecharts
!pip install pyecharts-snapshot

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached prettytable-3.17.0-py3-none-any.whl.metadata (34 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached prettytable-3.17.0-py3-none-any.whl (34 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pyecharts]/5 [pyecharts]
  Using cached pyecharts_snapshot-0.2.0-py2.py3-none-any.whl.metadata (12 kB)
  Using cached pyppeteer-2.0.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached pyee-11.1.1-py3-none-any.whl.metadata (2.8 kB)
  Using cached urllib3-1.26.20-py2.py3-none-any.whl.metadata (50 kB)
Using cached pyecharts_snapshot-0.2.0-py2.py3-none-any.whl (7.4 kB)
Using cached pyppeteer-2.0.0-py3-none-any.whl (82 kB)
Using cached appdirs-1.4.4-py2.py3-none-any.whl (9.6 kB)
Using cached pyee-11.1.1-py3-none-any.whl (15 kB)
Using cached urllib3-1.26.20-py2.py3-none-any.whl (144 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pyecha

In [10]:
import json
import re
from collections import Counter
from pyecharts import options as opts
from pyecharts.charts import Sankey

# Path relative to this notebook (search_result/ directory)
filepath = "ris/refined/classification_results_deepseek_2.json"

# Load JSON (robust to entries where classification may be missing)
with open(filepath, 'r', encoding='utf-8') as f:
    data = json.load(f)

pairs = Counter()
for item in data:
    # Prefer structured `classification` field if available
    domain = None
    tech = None
    cls = item.get('classification') if isinstance(item, dict) else None
    if isinstance(cls, dict):
        domain = cls.get('domain')
        if isinstance(domain, list):
            domain = domain[0] if domain else None
        tech = cls.get('technology')
        if isinstance(tech, list):
            tech = tech[0] if tech else None
    else:
        # try to parse raw_response JSON string if present
        rr = item.get('raw_response') if isinstance(item, dict) else None
        if isinstance(rr, str):
            try:
                parsed = json.loads(rr)
                domain = parsed.get('domain')
                tech = parsed.get('technology')
            except Exception:
                pass

    # normalize and filter
    # print(domain, tech)
    if domain and tech:
        domain = re.sub('\s+', ' ', domain).strip()
        tech = re.sub('\s+', ' ', tech).strip()
        pairs[(domain, tech)] += 1

# --- Top N Filtering ---
TOP_N = 10

# Count total occurrences for each domain and technology
domain_totals = Counter()
tech_totals = Counter()
for (d, t), v in pairs.items():
    domain_totals[d] += v
    tech_totals[t] += v

# Get top N domains and technologies
top_domains = set(d for d, _ in domain_totals.most_common(TOP_N))
top_techs = set(t for t, _ in tech_totals.most_common(TOP_N))

# Filter pairs to only include top domains and technologies
filtered_pairs = Counter()
for (d, t), v in pairs.items():
    if d in top_domains and t in top_techs:
        filtered_pairs[(d, t)] += v

pairs = filtered_pairs

# Prepare nodes and links for Sankey
domains = set()
techs = set()
for (d, t), v in pairs.items():
    domains.add(d)
    techs.add(t)

nodes = []
for d in sorted(domains):
    nodes.append({"name": d})
for t in sorted(techs):
    nodes.append({"name": t})

# build links with counts
links = []
for (d, t), v in pairs.items():
    links.append({"source": d, "target": t, "value": v})


c = (
    Sankey().add(
        "",
        nodes,
        links,
        linestyle_opt=opts.LineStyleOpts(opacity=0.2, curve=0.5, color="source"),
        label_opts=opts.LabelOpts(position="right", formatter="{b}: {c}",text_border_width=0,),
    ).set_global_opts(title_opts=opts.TitleOpts(title=f""))
) 

c.render_notebook()
    

In [11]:
final_domain_weights = Counter()
final_tech_weights = Counter()
for (d, t), v in filtered_pairs.items():
    final_domain_weights[d] += v
    final_tech_weights[t] += v

sorted_domains = [d for d, _ in final_domain_weights.most_common()]
sorted_techs = [t for t, _ in final_tech_weights.most_common()]

nodes = []
for d in sorted_domains:
    nodes.append({"name": d})
for t in sorted_techs:
    nodes.append({"name": t})

links = []
for (d, t), v in filtered_pairs.items():
    links.append({"source": d, "target": t, "value": v})

c = (
    Sankey().add(
        "",
        nodes,
        links,
        linestyle_opt=opts.LineStyleOpts(opacity=0.2, curve=0.5, color="source"),
        label_opts=opts.LabelOpts(position="right", formatter="{b}: {c}", text_border_width=0),
        node_gap=20,
    ).set_global_opts(title_opts=opts.TitleOpts(title="Domain to Technology Sankey (Sorted by Size)"))
)

c.render_notebook()

In [12]:
nodes

[{'name': 'Software Engineering'},
 {'name': 'Scientific Research'},
 {'name': 'Education'},
 {'name': 'Robotics (Embodied Intelligence)'},
 {'name': 'Healthy Care'},
 {'name': 'Manufacturing'},
 {'name': 'Society Simulation'},
 {'name': 'Annotation / Data Labeling'},
 {'name': 'Cybersecurity'},
 {'name': 'Industry 5.0'},
 {'name': 'Prompt Engineering'},
 {'name': 'Multi-Agent'},
 {'name': 'finetune'},
 {'name': 'multimodal'},
 {'name': 'RAG'},
 {'name': 'Reasoning'},
 {'name': 'Tool Use'},
 {'name': 'Other'},
 {'name': 'GraphRAG'},
 {'name': 'Large Language Models'}]

In [13]:
c.render("sankey.html")

'/Users/a/Library/CloudStorage/OneDrive-个人/Documents/phd/Review/1.96/search_result/sankey.html'